# 03b. 기준 ①② 판정과 사람 확인 재계산 (판정기준서 v1.1, D64, 2026-11-23)

- 구현: `src/criteria12.py` (판정기준서 1·2장), 규칙 C1-SCORE, C2-*, TR-C2B (`src/rules/rules.csv`), P5 보충 (`src/judge.py`).
- 대조: `data/review/기준12_참고결과.csv` (Claude 계산 참고 결과). 다르면 차이만 보고한다.
- 판정 없이 사실만 둔다. 원문은 출력하지 않는다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220)
from src.load_cot import load_cot, load_products
from src.judge import judge
cot = load_cot(); prod = load_products()
old, _ = judge(cot, prod, as_of="2026-11-18")
j, ctx = judge(cot, prod, as_of="2026-11-23")
c = ctx["cots"]

## 1. 기준 ① 항목별 확인 비율, 확인 수 분포

In [ ]:
items = [k for k in c.columns if k.startswith("①") and k not in ("① 확인 수", "① 법정 미확인")]
print(pd.DataFrame({k: c.groupby("split")[k].agg(lambda s: f"{(s != '미확인').mean() * 100:.1f}%") for k in items}).T)
print(pd.crosstab(c["① 확인 수"], c["split"], margins=True))

## 2. 기준 ② 분류 (Training / Validation)

In [ ]:
print(pd.crosstab(j["기준② 판정"], j["분할"], margins=True))
a = c[c["② 분류"] == "②-a 근거 부족 확정"]
print("②-a 미확인 항목:", a["① 법정 미확인"].str.split("|").explode().value_counts().to_dict())

## 3. 참고 결과와 대조 (차이만)

In [ ]:
ref = pd.read_csv(ROOT / "data" / "review" / "기준12_참고결과.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
ref["cot_id"] = ref["cot_id"].astype(int); ref = ref.set_index("cot_id").reindex(c.index)
NAME = {"목적": "취득 목적", "재산": "재산상황", "경험": "취득 경험"}
pairs = {"①취득 목적": "목적", "①재산상황": "재산", "①취득 경험": "경험", "①상품이해도": "이해도", "①위험에 대한 태도": "위험태도",
         "①연령": "연령", "성향_P5": "성향", "성향 출처": "성향출처", "고객 진술 성향": "고객진술", "② 분류": "기준2"}
diff = {a: int((c[a].astype(str) != ref[b].astype(str)).sum()) for a, b in pairs.items()}
print("항목별 차이 CoT 수:", diff)
for a, b in pairs.items():
    if diff[a]:
        d = c.index[c[a].astype(str) != ref[b].astype(str)]
        print(f"[{a}] 행=구현, 열=참고"); print(pd.crosstab(c.loc[d, a], ref.loc[d, b]))
d2 = c.index[c["② 분류"] != ref["기준2"]]
print("② 분류가 다른 cot_id:", sorted(d2.tolist()))
print("판정기준서 2장 표(3,941 / 2,532 / 537 / 191)와 구현:", c["② 분류"].value_counts().to_dict())

## 4. 사람 확인 재계산 (규칙만 기준) — ②-b 반영, P5로 성향이 정해진 CoT는 규칙으로 범위 재판정

In [ ]:
before = old["이관 판정"].eq("사람 확인"); after = j["이관 판정"].eq("사람 확인")
print("이관 사람 확인: 2026-11-18", before.sum(), "→ 2026-11-23", after.sum())
print("변경 전 '성향 불명' 사유:", (old["사람 확인 사유"] == "성향 불명(D21 비교 불가)").sum())
moved = old.loc[before & ~after.values, "cot_id"]
print("빠진 CoT의 새 이관 판정:", j.set_index("cot_id").loc[moved, "이관 판정"].value_counts().to_dict())
print("남은 사람 확인 사유:", j.loc[after, "사람 확인 사유"].value_counts().to_dict())
print("③ 범위 표기 손상(사람 확인):", (j["기준③ 판정"] == "판정 불가(범위 표기 손상)").sum())
p5 = j[j["P5 보충"]]
print("P5 보충 CoT:", len(p5), p5["cot_id"].tolist())
cmp = old.set_index("cot_id").loc[p5["cot_id"], ["기준③ 판정", "이관 판정"]].join(
    p5.set_index("cot_id")[["성향_P5", "기준③ 판정", "사유 코드", "이관 판정"]], lsuffix=" 전", rsuffix=" 후")
print(cmp.to_string())
print("제외 CoT 수: 전", (old["처리"] == "제외").sum(), "/ 후", (j["처리"] == "제외").sum())
x = j[j["성향_P5"].isin(["복수", "없음"]) & ~j["기준③ 판정"].str.startswith("판정 불가")]
print("P5는 복수·없음인데 D21로 단일 성향이 있어 ③이 판정된 CoT:", len(x), "/ ③:", x["기준③ 판정"].value_counts().to_dict())

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과